# Notebook 6 — Comprensione avanzada de immagini

**Curso:** EITC/AI/GVAPI — European IT Certification, Google Vision API
**Lezione 6:** Advanced Images Understanding (monumentos, logotipos, oggetti y Safe Search)

In [ ]:
import os
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "ruta/a/tu/clave.json"

from google.cloud import vision
client = vision.ImageAnnotatorClient()
print("Cliente listo.")


## 1. Rilevamento di monumenti (`landmark_detection`)

`landmark_detection` riconosce strutture naturali o costruite dall'uomo con rilevanza culturale, storica o geografica — la Torre Eiffel, la Statua della Libertà, il Taj Mahal. Funziona anche con il monumento parzialmente nascosto, ad angolazioni inusuali o con poca luce.

In [ ]:
def detect_landmark(file_path):
    with open(file_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.landmark_detection(image=image)

    for annotation in response.landmark_annotations:
        print("Monumento:", annotation.description)
        print("Ubicación:", annotation.locations)
        print("Confianza:", annotation.score)

    return response.landmark_annotations


# detect_landmark("torre_eiffel.jpg")


## Ejercicio 1 — Landmarks en pandas, ordenados por confidenza

Completa `landmarks_en_dataframe` para que procese una **lista** de percorsos de immagine, llame a `detect_landmark` en cada una, y consolide todos los risultati en un único DataFrame de pandas con columnas `immagine`, `description` y `score`, ordenado de mayor a menor confidenza.

In [ ]:
import pandas as pd

def landmarks_en_dataframe(rutas_imagenes):
    registros = []

    for ruta in rutas_imagenes:
        anotaciones = detect_landmark(ruta)
        # TODO: añade a "registros" un diccionario por cada anotación con las claves
        # "imagen", "description" y "score"
        ...

    df = pd.DataFrame(registros)
    # TODO: ordena el DataFrame por "score" de mayor a menor
    df = ...
    return df


# landmarks_en_dataframe(["torre_eiffel.jpg", "estatua_libertad.jpg"])


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
import pandas as pd

def landmarks_en_dataframe(percorsos_immaginees):
    registros = []

    for percorso in percorsos_immaginees:
        anotaciones = detect_landmark(percorso)
        for anotacion in anotaciones:
            registros.append({
                "immagine": percorso,
                "description": anotacion.description,
                "score": anotacion.score,
            })

    df = pd.DataFrame(registros)
    df = df.sort_values("score", ascending=False)
    return df


landmarks_en_dataframe(["torre_eiffel.jpg", "estatua_libertad.jpg"])
```

</details>

## 2. Detección de logotipos (`logo_detection`)

`logo_detection` identifica marcas y organizaciones dentro de una immagine, incluso con el logotipo parcialmente oculto o distorsionado. En una test descrita en el material del curso, la API reconoció sin problema los logotipos de Apple, Nintendo, Best Buy y Nike, pero no logró identificar los de IBM, Circuit City y Playboy — la precisión global rondó el 90%.

In [ ]:
def detect_logos(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.logo_detection(image=image)
    logos = response.logo_annotations

    for logo in logos:
        print("Logo:", logo.description)
        print("Confianza:", logo.score)
        print("Cuadro delimitador:", logo.bounding_poly)

    return logos


# detect_logos("tienda_electronica.jpg")


## Ejercicio 2 — Estudio de falsos negativos

Completa `tasa_deteccion` para que, dada una lista de percorsos de immagini que contienen *un* logo conocido cada una y una lista paralela con el nombre esperado del logo, calcule el porcentaje de aciertos (immagini en las que `logo_detection` detectó correctamente el logo esperado, comparando en minúsculas).

In [ ]:
def tasa_deteccion(rutas_imagenes, nombres_esperados):
    aciertos = 0

    for ruta, esperado in zip(rutas_imagenes, nombres_esperados):
        logos = detect_logos(ruta)
        # TODO: comprueba si "esperado" aparece entre los logos detectados (compara en minúsculas)
        detectado = ...
        if detectado:
            aciertos += 1

    return aciertos / len(rutas_imagenes) * 100


# tasa_deteccion(["gap.jpg", "nike.jpg", "audi.jpg"], ["gap", "nike", "audi"])


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
def tasa_deteccion(percorsos_immaginees, nombres_esperados):
    aciertos = 0

    for percorso, esperado in zip(percorsos_immaginees, nombres_esperados):
        logos = detect_logos(percorso)
        rilevato = any(logo.description.lower() == esperado.lower() for logo in logos)
        if rilevato:
            aciertos += 1

    return aciertos / len(percorsos_immaginees) * 100


tasa_deteccion(["gap.jpg", "nike.jpg", "audi.jpg"], ["gap", "nike", "audi"])
```

</details>

## 3. Detección de oggetti (`object_localization`)

`object_localization` combina dos tareas: **clasificación** (qué es cada oggetto) y **localización** (dónde está, con su riquadro delimitador). Internamente usa un modello **SSD** (detector multibox de disparo único), una arquitectura de red neuronal convolucional (CNN) entrenada con millones de immagini.

Fíjate en que `bounding_poly.normalized_vertices` devuelve coordenadas **normalizadas** entre cero y uno; para dibujarlas sobre la immagine real tendrás que multiplicarlas por el ancho y el alto (lo verás en el notebook 8, con Pillow).

In [ ]:
def detect_objects(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.object_localization(image=image)
    objects = response.localized_object_annotations

    for obj in objects:
        print(f"{obj.name} (confianza: {obj.score:.2f})")

    return objects


# detect_objects("salon.jpg")


## Ejercicio 3 — Efecto de la orientación en la confidenza

Completa `comparar_angulos` para que, dado un diccionario `{"angulo": "percorso_immagine"}` de la misma escena fotografiada desde distintos ángulos, construya un DataFrame con la confidenza media de todos los oggetti rilevati en cada ángulo, para comprobar si la orientación afecta a la fiabilidad de la rilevamento.

In [ ]:
import pandas as pd

def comparar_angulos(imagenes_por_angulo):
    registros = []

    for angulo, ruta in imagenes_por_angulo.items():
        objetos = detect_objects(ruta)
        # TODO: calcula la confianza media de "objetos" (usa 0 si la lista está vacía)
        confianza_media = ...
        registros.append({"angulo": angulo, "confianza_media": confianza_media})

    return pd.DataFrame(registros)


# comparar_angulos({"frontal": "espejo_frontal.jpg", "lateral": "espejo_lateral.jpg"})


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
import pandas as pd

def comparar_angulos(immaginees_por_angulo):
    registros = []

    for angulo, percorso in immaginees_por_angulo.items():
        oggetti = detect_objects(percorso)
        scores = [obj.score for obj in oggetti]
        confidenza_media = sum(scores) / len(scores) if scores else 0
        registros.append({"angulo": angulo, "confidenza_media": confidenza_media})

    return pd.DataFrame(registros)


comparar_angulos({"frontal": "espejo_frontal.jpg", "lateral": "espejo_lateral.jpg"})
```

</details>

## 4. Detección de contenuto explícito (`safe_search_detection`)

`safe_search_detection` estima la probabilidad de que una immagine contenga contenuto inapropiado, en cinco categorías: **adult** (contenuto sexual explícito), **spoof** (immagini manipuladas o engañosas), **medical** (contenuto médico o anatómico), **violence** (violencia) y **racy** (contenuto sugerente).

Cada categoría devuelve un valor de la misma enumeración de verosimilitud que viste en la rilevamento de volti: `UNKNOWN`, `VERY_UNLIKELY`, `UNLIKELY`, `POSSIBLE`, `LIKELY` o `VERY_LIKELY`.

**Ten presente sus límites**: Safe Search puede producir falsos positivos y falsos negativos. Combínalo siempre con otras técnicas de moderación —revisión humana, reportes de la comunidad usuaria— en lugar de confiar en él come única barrera.

In [ ]:
LIKELIHOOD_NAMES = (
    "UNKNOWN",
    "VERY_UNLIKELY",
    "UNLIKELY",
    "POSSIBLE",
    "LIKELY",
    "VERY_LIKELY",
)

def check_safe_search(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.safe_search_detection(image=image)
    safe_search = response.safe_search_annotation

    return {
        "adult": LIKELIHOOD_NAMES[safe_search.adult],
        "spoof": LIKELIHOOD_NAMES[safe_search.spoof],
        "medical": LIKELIHOOD_NAMES[safe_search.medical],
        "violence": LIKELIHOOD_NAMES[safe_search.violence],
        "racy": LIKELIHOOD_NAMES[safe_search.racy],
    }


# print(check_safe_search("imagen.jpg"))


## Esercizio 4 — Regola di moderazione automatica

Completa `requiere_revision` affinché restituisca `True` se una delle cinque categorie di Safe Search raggiunge `LIKELY` o `VERY_LIKELY` (indice quattro o cinque in `LIKELIHOOD_NAMES`), e `False` in caso contrario.

In [ ]:
def requiere_revision(image_path):
    resultado = check_safe_search(image_path)

    # TODO: comprueba si algún valor de "resultado" es "LIKELY" o "VERY_LIKELY"
    revisar = ...
    return revisar


# print(requiere_revision("imagen.jpg"))


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
def requiere_revision(image_path):
    risultato = check_safe_search(image_path)
    livelli_altos = {"LIKELY", "VERY_LIKELY"}
    revisar = any(valor in livelli_altos for valor in risultato.values())
    return revisar


print(requiere_revision("immagine.jpg"))
```

</details>

---

**Siguiente paso:** abre `07_Dati_Web.ipynb`.